1) Создать качественный датасет для учителя-LLM
2) Создать качественный датасет для ученика-LLM
3) Обучить ученика-LLM

20К - для обучения
5K - для тестов

In [2]:
!pip install sentence_transformers
!pip install langchain_text_splitters
!pip install transformers torch accelerate bitsandbytes
!pip install rank_bm25

# !pip install faiss-gpu-cu12 # *
!pip install faiss-cpu # if * doesnt work locally

In [3]:
!git clone https://github.com/chipslays/russian-recipes-parser.git

Cloning into 'russian-recipes-parser'...
remote: Enumerating objects: 14584, done.
remote: Counting objects: 100% (14584/14584), done.
remote: Compressing objects: 100% (1592/1592), done.
remote: Total 14584 (delta 12973), reused 14580 (delta 12971), pack-reused 0 (from 0)
Receiving objects: 100% (14584/14584), 18.55 MiB | 16.41 MiB/s, done.
Resolving deltas: 100% (12973/12973), done.


In [1]:
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [1]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import sentence_transformers.losses as losses
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
import re
from huggingface_hub import login

import json
import glob
from io import StringIO
import gc
from typing import Literal, Optional, Callable
from tqdm import tqdm
from numpy.typing import NDArray
import random
import copy
import datetime
import math
import traceback
import matplotlib.pyplot as plt

In [3]:
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device: Literal['mps', 'cpu'] = 'mps' if torch.backends.mps.is_available() else 'cpu'
device = 'cpu'
batch_size = 6
print(f"Используем устройство: {device}")

Используем устройство: cpu


In [ ]:
COLAB_HUGGING_FACE_ACCESS_TOKEN = "" # your token
login(token=COLAB_HUGGING_FACE_ACCESS_TOKEN)

In [5]:
with open('recipes.json', 'r', encoding='utf-8') as f:
    parsed_recipe_list = json.load(f)

print(f"Загружено {len(parsed_recipe_list)} рецептов")

Загружено 13702 рецептов


In [7]:
recipe_list_path = "russian-recipes-parser/storage/recipes/[0-9][0-9][0-9]/*.json"
parsed_recipe_list = []

recipe_file_list = glob.glob(recipe_list_path)
print(f"Найдено {len(recipe_file_list)} файлов рецептов для обработки.")
for step_index, filepath in tqdm(enumerate(recipe_file_list), total=len(recipe_file_list), desc="Обработка рецептов"):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            recipe_data = json.load(f)
        parsed_recipe = {
            "name": recipe_data.get('title', ''),
            "ingredients": recipe_data.get('ingredients', []),
            "instruction": recipe_data.get('instruction', []),
            "full_text": "",
        }
        full_recipe_text = StringIO()
        recipe_title = f"Название рецепта: {parsed_recipe['name']}\n\n"
        full_recipe_text.write(recipe_title)
        ingredients_data_list = parsed_recipe['ingredients']
        if isinstance(ingredients_data_list, list) and ingredients_data_list:
            ingredients_data = ingredients_data_list[0]
            full_recipe_text.write(f"{ingredients_data.get('name', '')}:\n")
            for step_index, ingredient in enumerate(ingredients_data.get('list', [])):
                full_recipe_text.write(f"{step_index + 1}) {ingredient.get('name', '')} {ingredient.get('value', '')} {ingredient.get('type', '')}\n")
        full_recipe_text.write(f"\nИнструкции по приготовлению:\n")
        for step_index, instruction_data in enumerate(recipe_data.get('instruction', [])):
          full_recipe_text.write(f"{step_index + 1}) {instruction_data.get('text', '')}\n")
        parsed_recipe["full_text"] = full_recipe_text.getvalue()
        parsed_recipe_list.append(parsed_recipe)
        if step_index and step_index % 1000 == 0:
            gc.collect()
    except Exception as e:
        print(f"Ошибка при обработке файла {filepath}: {e}")

print(f"Загружено {len(parsed_recipe_list)} рецептов.")


Найдено 13702 файлов рецептов для обработки.


Обработка рецептов: 100%|██████████| 13702/13702 [00:01<00:00, 9798.02it/s]

Загружено 13702 рецептов.


In [ ]:
print("\n--- Пример рецепта ---")
for retriever_index in range(5):
    parsed_recipe = parsed_recipe_list[retriever_index]
    print(parsed_recipe["full_text"])


--- Пример рецепта ---
Название рецепта: Плов с маринованной говядиной в мультиварке

Ингредиенты на 6 порции:
1) говядина 550 г
2) рис длиннозерный 370 г
3) лук белый 3 шт.
4) морковь 3 шт.
5) чеснок 4-5 зубчиков
6) масло растительное None None
7) приправы None None
8) соль None None

Инструкции по приготовлению:
1) Основной продукт очистить от пленок, нарезать кусками.
2) Мясо залить растительным маслом с приправами на два часа.
3) Влить растительное масло в чашу мультиварки, выставить режим «Жарка» на сорок минут.
4) В течение половины этого времени готовить мясо.
5) Далее всыпать белый лук, нарезанный кубиками, бруски сладкой моркови.
6) Все вместе жарить до окончания программы без крышки.
7) Влить горячую воду, всыпать соль и специи, нарезанный чеснок, выставить режим «Тушение» на пятьдесят минут.
8) В последнюю очередь всыпать промытый рис, влить полтора стакана горячей воды.
9) Продукты не перемешивать.
10) Блюдо тушить еще сорок минут.

Название рецепта: Плов с куриным филе и 

In [30]:
output_filename = 'recipes.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(parsed_recipe_list, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(parsed_recipe_list)} рецептов в {output_filename}")


Сохранено 13702 рецептов в recipes.json


In [6]:
with open('recipe_chunks.json', 'r', encoding='utf-8') as f:
    train_chunk_list = json.load(f)

print(f"Загружено {len(train_chunk_list)} чанков")

Загружено 95516 чанков


In [8]:
class RecipeChunker:
    def chunk_recipe(self, parsed_recipe, max_chunk_size=512):
      chunk_list = []
      parsed_recipe_name = parsed_recipe['name']

      # Уровень 1: Целый рецепт (глобальный контекст)
      chunk_list.append({
        'recipe_name': parsed_recipe_name,
        'scale': 'full',
        'text': parsed_recipe['full_text'],
      })

      # Уровень 2: Секции (ингредиенты)
      if parsed_recipe['ingredients']:
        ingredient_list = parsed_recipe['ingredients'][0]
        ingredient_text = StringIO()
        ingredient_text.write(f"Ингредиенты для рецепта \"{parsed_recipe_name}\":\n")
        for step_index, ingredient in enumerate(ingredient_list['list']):
          ingredient_text.write(f"{step_index + 1}) {ingredient['name']} {ingredient['value']} {ingredient['type']}\n")
        chunk_list.append({
          'recipe_name': parsed_recipe_name,
          'scale': 'ingredients',
          'text': ingredient_text.getvalue()
        })

      # Уровень 3: Отдельные шаги
      if parsed_recipe['instruction']:
        splitter = RecursiveCharacterTextSplitter(
          chunk_size=max_chunk_size,
          chunk_overlap=30,
          separators=[". ", ", ", " "]
        )
        instruction_text_prefix = f"Инструкции для рецепта \"{parsed_recipe_name}\":\n"
        for step_index, instruction_data in enumerate(parsed_recipe['instruction']):
          step_number = step_index + 1
          instruction_text = StringIO()
          instruction_text.write(instruction_text_prefix)
          instruction_text.write(f"{step_number}) {instruction_data['text']}\n")
          if instruction_text.tell() <= max_chunk_size:
            chunk_list.append({
              'recipe_name': parsed_recipe_name,
              'scale': 'step',
              'step_number': step_number,
              'text': instruction_text.getvalue(),
            })
            continue
          instruction_text_chunk_list = splitter.split_text(instruction_text.getvalue())
          for substep_index, instruction_text_chunk in enumerate(instruction_text_chunk_list):
            chunk_list.append({
              'recipe_name': parsed_recipe_name,
              'scale': 'substep',
              'text': instruction_text_chunk,
              'step_number': step_number,
              'substep_number': substep_index + 1,
            })
      return chunk_list


In [ ]:
recipe_chunker = RecipeChunker()
train_chunk_list = []
for retriever_index, parsed_recipe in tqdm(enumerate(parsed_recipe_list), total=len(parsed_recipe_list), desc="Создание чанков рецептов"):
  train_chunk_list.extend(recipe_chunker.chunk_recipe(parsed_recipe))
  if retriever_index and retriever_index % 1000 == 0:
    gc.collect()

print(f"Создано {len(train_chunk_list)} чанков разного масштаба")
for retriever_index in range(5):
  eval_batch = train_chunk_list[retriever_index]
  print(json.dumps(eval_batch, ensure_ascii=False, indent=2))

Создание чанков рецептов: 100%|██████████| 13702/13702 [00:06<00:00, 2074.33it/s]

Создано 95516 чанков разного масштаба
{
  "recipe_name": "Плов с маринованной говядиной в мультиварке",
  "scale": "full",
  "text": "Название рецепта: Плов с маринованной говядиной в мультиварке\n\nИнгредиенты на 6 порции:\n1) говядина 550 г\n2) рис длиннозерный 370 г\n3) лук белый 3 шт.\n4) морковь 3 шт.\n5) чеснок 4-5 зубчиков\n6) масло растительное None None\n7) приправы None None\n8) соль None None\n\nИнструкции по приготовлению:\n1) Основной продукт очистить от пленок, нарезать кусками.\n2) Мясо залить растительным маслом с приправами на два часа.\n3) Влить растительное масло в чашу мультиварки, выставить режим «Жарка» на сорок минут.\n4) В течение половины этого времени готовить мясо.\n5) Далее всыпать белый лук, нарезанный кубиками, бруски сладкой моркови.\n6) Все вместе жарить до окончания программы без крышки.\n7) Влить горячую воду, всыпать соль и специи, нарезанный чеснок, выставить режим «Тушение» на пятьдесят минут.\n8) В последнюю очередь всыпать промытый рис, влить полт

In [ ]:
output_filename = 'recipe_chunks.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(train_chunk_list, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(train_chunk_list)} чанков в {output_filename}")


Сохранено 95516 чанков в recipe_chunks.json


In [ ]:
# 3352
class RecipeQuestionGenerator:
    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct"):
        # TODO: it needs to read about it
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="cuda:0",
            trust_remote_code=True,
            use_cache=True
        )
        self.model.eval()
        self.model = torch.compile(self.model, mode="reduce-overhead")
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token


    def generate_question_list(self, recipe_text: str, question_amount=3) -> list[str]:
        question_prompt = self._create_question_prompt(recipe_text, question_amount)
        tokenizer_output = self.tokenizer(question_prompt, return_tensors="pt", truncation=True, max_length=2048).to(self.model.device)
        with torch.inference_mode():
            model_output = self.model.generate(
                **tokenizer_output,
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                top_k=50,
                repetition_penalty=1.05
            )
        model_response = self.tokenizer.decode(model_output[0], skip_special_tokens=True)
        return self._extract_question_list(model_response, question_amount)


    def _extract_question_list(self, model_response: str, question_amount: int) -> list[str]:
        question_list = []
        question = ""
        if "Вопрос 1:" in model_response:
            model_response = model_response.split("Вопрос 1:")[-1]
            for index in range(1, question_amount + 1):
                current_marker = f"Вопрос {index}:"
                next_marker = f"Вопрос {index+1}:" if index + 1 <= question_amount else None
                if next_marker and next_marker in model_response:
                    question, model_response = model_response.split(next_marker, 1)
                    question = question.replace(current_marker, "").strip()
                else:
                    question = model_response.replace(current_marker, "").strip()
                    if index == question_amount - 1:
                        question = model_response.strip()
                question_list.append(question)
        question_list = [question.strip() for question in question_list if question and '?' in question]
        return question_list[:question_amount]

    def _create_question_prompt(self, recipe_text: str, question_amount: int = 3) -> str:
        return f"""Ты — ассистент, который помогает обучать систему поиска рецептов. Твоя задача — придумать {question_amount} разнообразных вопросов, на которые отвечает данный текст.

Правила:
1. Вопросы должны быть краткими и естественными (как их мог бы задать пользователь)
2. Каждый вопрос должен быть уникальным и затрагивать разные аспекты текста
3. Используй информацию ТОЛЬКО из текста
4. Не задавай вопросы, требующие знаний вне текста

Текст:
{recipe_text}

Отвечай строго в формате:
Вопрос 1: [первый вопрос]
Вопрос 2: [второй вопрос]
Вопрос 3: [третий вопрос]

Не добавляй ничего кроме вопросов в этом формате."""


In [ ]:
question_generator = RecipeQuestionGenerator()
question_list = []
part_of_chunk_list = train_chunk_list[3000:6000]
for retriever_index, eval_batch in tqdm(enumerate(part_of_chunk_list), total=len(part_of_chunk_list), desc="Генерация вопросов для чанков рецептов"):
  if eval_batch['scale'] != 'full':
    continue
  eval_batch['questions'] = question_generator.generate_question_list(eval_batch['text'])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Генерация вопросов для чанков рецептов: 100%|██████████| 3000/3000 [51:22<00:00,  1.03s/it]  


In [ ]:
output_filename = 'recipe_chunks_with_questions_2.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(part_of_chunk_list, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(part_of_chunk_list)} чанков в {output_filename}")


In [5]:
input_filename = 'recipe_chunks_with_questions_1.json'
with open(input_filename, 'r', encoding='utf-8') as f:
    train_chunk_list = json.load(f)

print(f"Загружено {len(train_chunk_list)} чанков")


Загружено 3000 чанков


In [6]:
input_filename = 'recipe_chunks_with_questions_2.json'
with open(input_filename, 'r', encoding='utf-8') as f:
    eval_chunk_list = json.load(f)
print(f"Загружено {len(eval_chunk_list)} чанков")

Загружено 1499 чанков


In [7]:
class BM25Model:
    def __init__(self, chunk_list: list):
        text_chunk_list = [chunk['text'] for chunk in chunk_list]
        token_list = []
        for chunk in text_chunk_list:
            word_list = re.findall(r'\w+', chunk.lower())
            token_list.append(word_list)
        self.model = BM25Okapi(token_list)


    def search(self, question: str, k: int) -> tuple[torch.Tensor, torch.Tensor]:
        tokenized_query = re.findall(r'\w+', question.lower())
        score_list = self.model.get_scores(tokenized_query)
        # TODO: dont like the copy
        index_list = torch.tensor(np.argsort(score_list)[-k:][::-1].copy())
        return index_list, torch.tensor(score_list)[index_list]

In [8]:
class DenseModel:
    def __init__(self, chunk_list: list, model_name="deepvk/USER2-small"):
        self.model = SentenceTransformer(model_name)
        text_chunk_list = [chunk['text'] for chunk in chunk_list]
        self.embedding_list = self.model.encode(
            text_chunk_list,
            prompt_name="search_document",
            show_progress_bar=True,
            batch_size=batch_size
        )
        faiss.normalize_L2(self.embedding_list)
        self.index = faiss.IndexFlatIP(self.embedding_list.shape[1])
        self.index.add(self.embedding_list.astype('float32')) # type: ignore


    def search(self, question: str, k: int) -> tuple[torch.Tensor, torch.Tensor]:
        query_embedding = self.model.encode([question], prompt_name="search_query")
        faiss.normalize_L2(query_embedding)
        scores, indices = self.index.search(query_embedding.astype('float32'), k) # type: ignore
        return torch.tensor(indices[0]), torch.tensor(scores[0])

In [9]:
class TeacherRetrieverPool(nn.Module):
    def __init__(self, teacher_amount=2, top_k=12, rrf_K=60):
        super().__init__()
        self.RRF_K = rrf_K
        self.top_k = top_k
        self.total_teacher_amount = teacher_amount
        self.teacher_model_dict = {}
        self.weight_tensor = nn.Parameter(torch.ones(teacher_amount))


    def forward(
        self,
        query_data_batch: list[tuple[str, torch.Tensor]], # [(str, [current_teacher_amount, top_k])]
        encoder_device: Literal['mps', 'cpu', 'cuda']
    ):
        encoder_device = encoder_device or next(self.parameters()).device
        batch_size = len(query_data_batch)
        query_list = []    
        fuse_ranked_index_tensor_list = []

        for query_data in query_data_batch:
            query_list.append(query_data[0])
            ranking_tensor = self.get_fuse_ranked_index_tensor(query_data[1])
            fuse_ranked_index_tensor_list.append(ranking_tensor)
        fused_ranked_index_batch = torch.stack(fuse_ranked_index_tensor_list)  # [batch, top_k]
        
        positive_document_position_tensor = torch.randint(0, 10, (batch_size,))
        positive_document_index_tensor = fused_ranked_index_batch[torch.arange(batch_size), positive_document_position_tensor]
        top_k = query_data_batch[0][1].size(1)
        hard_negative_position_tensor = (top_k // 2 - 1) + torch.randint(0, 5, (batch_size,))
        hard_negative_document_index_tensor = fused_ranked_index_batch[torch.arange(batch_size), hard_negative_position_tensor].unsqueeze(1)  # [batch, 1]
    
        mask = ~torch.eye(batch_size, dtype=torch.bool)
        other_positive_document_index_tensor = positive_document_index_tensor.unsqueeze(0).expand(batch_size, batch_size)[mask].view(batch_size, -1)

        # TODO: remove duplicates if they are 
        hard_negative_document_index_tensor = torch.cat([hard_negative_document_index_tensor, other_positive_document_index_tensor], dim=1)  # [batch, 1 + (batch-1)]
        return query_list, positive_document_index_tensor.to(encoder_device), hard_negative_document_index_tensor.to(encoder_device)


    def add_teacher(self, teacher_name: str, teacher_model):
        if len(self) < self.total_teacher_amount:
            self.teacher_model_dict[teacher_name] = teacher_model


    def get_ranking_tensor(self, question_list: list[str]) -> torch.Tensor: # [teacher_amount, top_k]
        ranking_tensor = torch.zeros((self.total_teacher_amount, self.top_k))
        for question in question_list:
            for index, (_, teacher_model) in enumerate(self.teacher_model_dict.items()):
                _, score_list = teacher_model.search(question, self.top_k)
                ranking_tensor[index, :] = score_list
        return ranking_tensor


    # ranking_tensor shape is [current_teacher_amount, top_k]
    def get_fuse_ranked_index_tensor(self, ranking_tensor: torch.Tensor) -> torch.Tensor:
        if ranking_tensor.size(0) > self.total_teacher_amount:
            raise Exception("ranking_tensor.size(0) > total_teacher_amount")
        rrf_score_tensor = self.weight_tensor[:ranking_tensor.size(0)] @ (1.0/(self.RRF_K + ranking_tensor)) # [total_teacher_amount] @ [current_teacher_amount, top_k] -> [top_k]
        sorted_index_tensor = torch.argsort(rrf_score_tensor, descending=True)
        return sorted_index_tensor
    

    def __len__(self) -> int:
        return len(self.teacher_model_dict)


In [10]:
class MultiNegativeLoss(nn.Module):
    def __init__(self, encoder, temperature=0.05):
        super().__init__()
        self.encoder = encoder
        self.temperature = temperature


    def forward(self, sentence_features, labels=None):
        query_features, positive_features, negative_features = sentence_features
        query_embedding = self.encoder(query_features)["sentence_embedding"]              # [B, D]
        positive_chunk_embedding = self.encoder(positive_features)["sentence_embedding"]  # [B, D]
        negative_chunk_embedding = self.encoder(negative_features)["sentence_embedding"]  # [B*N, D]
        B = query_embedding.size(0)
        N = negative_chunk_embedding.size(0) // B
        negative_chunk_embedding = negative_chunk_embedding.view(B, N, -1)  # [B, N, D]
        query_embedding = F.normalize(query_embedding, dim=1)
        positive_chunk_embedding = F.normalize(positive_chunk_embedding, dim=1)
        negative_chunk_embedding = F.normalize(negative_chunk_embedding, dim=2)
        positive_scores = torch.sum(query_embedding * positive_chunk_embedding, dim=1, keepdim=True)  # [B,1]
        negative_scores = torch.bmm(negative_chunk_embedding, query_embedding.unsqueeze(2)).squeeze(2)  # [B,N]
        logits = torch.cat([positive_scores, negative_scores], dim=1) / self.temperature
        labels = torch.arange(B, dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)

In [11]:
class PovaryoshkaRetriever(nn.Module):
  def __init__(
    self,
    document_list: list[str],
    teacher_retriever_pool: TeacherRetrieverPool,
    encoder_name="deepvk/USER2-small",
    dtype="float32",
    temperature=0.05,
    matryoshka_dims=[384, 256, 128],
    nlist=100
  ):
    super().__init__()
    self.temperature = temperature
    self.document_list = document_list
    self.retriever_index = None
    self.DTYPE = dtype
    self.encoder = SentenceTransformer(encoder_name, model_kwargs={"dtype": dtype})
    base_encoder_loss = MultiNegativeLoss(self.encoder, temperature)
    self.encoder_loss = losses.MatryoshkaLoss(
      self.encoder,
      base_encoder_loss,
      matryoshka_dims
    )
    self.teacher_retriever_pool = teacher_retriever_pool
    self.NLIST = nlist


  def forward(
    self,
    batch_query_data: list[tuple[str, torch.Tensor]] # [(str, [current_teacher_amount, top_k])]
  ) -> torch.Tensor:
    assert self.document_list is not None, "Firstly, initialize document_list field"
    encoder_device = next(self.encoder.parameters()).device
    query_list, positive_chunk_index_list, negative_chunk_indexes_list = self.teacher_retriever_pool(
      batch_query_data,
      encoder_device
    )

    anchor_list = []
    positive_chunk_list = []
    negative_chunks_list = []

    for query_index in range(len(query_list)):
      query = query_list[query_index]
      positive_chunk = self.document_list[positive_chunk_index_list[query_index]]
      for negative_chunk_index in negative_chunk_indexes_list[query_index]:
        negative_chunk = self.document_list[negative_chunk_index]
        negative_chunks_list.append(negative_chunk)
      anchor_list.append(query)
      positive_chunk_list.append(positive_chunk)

    anchor_features = self.to_device(
      self.encoder.tokenize(anchor_list), encoder_device
    )
    positive_features = self.to_device(
      self.encoder.tokenize(positive_chunk_list), encoder_device
    )
    negative_features = self.to_device(
      self.encoder.tokenize(negative_chunks_list), encoder_device
    )
    return self.encoder_loss(
      sentence_features=[anchor_features, positive_features, negative_features],
      labels=None
    )


  def to_device(self, features, device):
    return {k: v.to(device) for k, v in features.items()}


  def get_encoded_query_tensor(
    self,
    query_list: list[str],
    is_normalize=True
  ) -> torch.Tensor:
    self.eval()
    with torch.inference_mode():
      return self._get_encoded_text_tensor(query_list, True, is_normalize)


  def get_encoded_document_tensor(
    self,
    document_list: list[str],
    is_normalize=True
  ) -> torch.Tensor:
    self.eval()
    with torch.inference_mode():
      return self._get_encoded_text_tensor(document_list, is_normalize)
      

  def _get_encoded_text_tensor(
    self,
    text_list: list[str],
    is_query=False,
    is_normalize=True
  ) -> torch.Tensor:
    embedding_text_list = self.encoder.encode(
      text_list,
      prompt_name="search_query" if is_query else "search_document",
      convert_to_tensor=True
    )
    if is_normalize:
      embedding_text_list = F.normalize(embedding_text_list, dim=1)
    return embedding_text_list


  def build_index(self):
    """Строит FAISS индекс для поиска"""
    self.eval()
    print(self.DTYPE)
    
    if not self.document_list:
      raise ValueError("document_list is empty!")
    
    print(f"📊 Индексация {len(self.document_list)} документов...")
    
    with torch.inference_mode():
      embedding_list = self._get_encoded_text_tensor(self.document_list)
        
      # Проверяем, что получили тензор
      if embedding_list is None or embedding_list.numel() == 0:
        raise ValueError("No embeddings generated!")
        
    # Перемещаем на CPU и преобразуем в numpy
    embedding_list = embedding_list.cpu().numpy()
    
    # ✅ FAISS требует float32
    embedding_list = embedding_list.astype('float32')
    
    # ✅ Убеждаемся, что массив непрерывный в памяти
    embedding_list = np.ascontiguousarray(embedding_list)
    
    embedding_dimension = embedding_list.shape[1]
    num_vectors = embedding_list.shape[0]
    
    print(f"   Размерность: {embedding_dimension}, векторов: {num_vectors}")
    
    # Создаем индекс
    retriever_index = faiss.IndexFlatIP(embedding_dimension)
    
    # Добавляем векторы
    retriever_index.add(embedding_list) # type: ignore
    
    self.retriever_index = retriever_index
    print(f"✅ Индекс построен: {self.retriever_index.ntotal} векторов")


  def search(
    self,
    query_list: list[str],
    top_k=15
  ) -> list[list[int]]:
    assert self.retriever_index is not None, "Firstly call build_index()"
    self.eval()
    with torch.inference_mode():
      query_embedding_list = self._get_encoded_text_tensor(
        query_list,
        True
      ).cpu().numpy().astype(self.DTYPE)
    _, indices_list = self.retriever_index.search(query_embedding_list, top_k) # type: ignore
    return indices_list


  def save(self, path: str):
    self.encoder.save(path)


  def load(self, path: str):
    self.encoder = SentenceTransformer(path)


In [12]:
pruned_train_chunk_list = []
for eval_batch in train_chunk_list:
    if 'questions' not in eval_batch:
        continue
    if not eval_batch['questions']:
        continue
    pruned_train_chunk_list.append(eval_batch)

pruned_eval_chunk_list = []
for eval_batch in eval_chunk_list:
    if 'questions' not in eval_batch:
        continue
    if not eval_batch['questions']:
        continue
    pruned_eval_chunk_list.append(eval_batch)


In [13]:
output_filename = 'pruned_recipe_chunks_with_questions_1.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(pruned_train_chunk_list, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(pruned_train_chunk_list)} чанков в {output_filename}")


Сохранено 406 чанков в pruned_recipe_chunks_with_questions_1.json


In [14]:
output_filename = 'pruned_recipe_chunks_with_questions_2.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(pruned_eval_chunk_list, f, ensure_ascii=False, indent=2)

print(f"Сохранено {len(pruned_eval_chunk_list)} чанков в {output_filename}")


Сохранено 173 чанков в pruned_recipe_chunks_with_questions_2.json


In [15]:
teacher_retriever_pool = TeacherRetrieverPool(top_k=10)
pruned_train_chunk_list = pruned_train_chunk_list[:10]
pruned_eval_chunk_list = pruned_eval_chunk_list[:10]
pruned_common_chunk_list = pruned_train_chunk_list + pruned_eval_chunk_list
teacher_retriever_pool.add_teacher('bm25', BM25Model(pruned_common_chunk_list))
teacher_retriever_pool.add_teacher('dense', DenseModel(pruned_common_chunk_list))


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Compiling the model with `torch.compile` and using a `torch.mps` device is not supported. Falling back to non-compiled mode.


In [16]:
# TODO: Think do we need here to add context ?
train_ranked_chunk_list = []
for eval_batch in pruned_train_chunk_list:
    # TODO: not only the first
    first_question = f'Контекст: {eval_batch['recipe_name']}\n\n{eval_batch['questions'][0]}'
    eval_batch['questions'][0] = first_question
    ranking_tensor = teacher_retriever_pool.get_ranking_tensor([first_question])
    eval_batch['ranking_tensors'] = [ranking_tensor]
    train_ranked_chunk_list.append(eval_batch)

eval_ranked_chunk_list = []
for eval_batch in pruned_eval_chunk_list:
    # TODO: not only the first
    first_question = f'Контекст: {eval_batch['recipe_name']}\n\n{eval_batch['questions'][0]}'
    eval_batch['questions'][0] = first_question
    ranking_tensor = teacher_retriever_pool.get_ranking_tensor([first_question])
    eval_batch['ranking_tensors'] = [ranking_tensor]
    eval_ranked_chunk_list.append(eval_batch)

torch.save(train_ranked_chunk_list, 'recipe_chunks_with_ranking_tensors_1.pth')
torch.save(eval_ranked_chunk_list, 'recipe_chunks_with_ranking_tensors_2.pth')

In [17]:
train_ranked_chunk_list = torch.load('recipe_chunks_with_ranking_tensors_1.pth')
for eval_batch in train_ranked_chunk_list:
    eval_batch['ranking_tensors'] = [ranking_tensor.to(device) for ranking_tensor in eval_batch['ranking_tensors']]
print(len(train_ranked_chunk_list))

10


In [18]:
eval_ranked_chunk_list = torch.load('recipe_chunks_with_ranking_tensors_2.pth')
for eval_batch in eval_ranked_chunk_list:
    eval_batch['ranking_tensors'] = [ranking_tensor.to(device) for ranking_tensor in eval_batch['ranking_tensors']]
print(len(eval_ranked_chunk_list))

10


In [19]:
class RetrieverDataset(Dataset):
    def __init__(self, ranked_chunk_list):
        self.ranked_chunk_list = ranked_chunk_list


    def __len__(self) -> int:
        return len(self.ranked_chunk_list)

    # TODO
    def __getitem__(self, index: int):
        ranked_chunk = self.ranked_chunk_list[index]
        return ranked_chunk['questions'][0], ranked_chunk['ranking_tensors'][0]


In [20]:
def create_povaryoshka_retriever_fn(teacher_retriever_pool: TeacherRetrieverPool):
    current_epoch_number = 0

    def povaryoshka_retriever_collate_fn(
        query_data_batch: list[tuple[str, torch.Tensor]] # [(str, [teacher_amount, top_k])]
    ) -> list[tuple[str, torch.Tensor]]: # [(str, [current_teacher_amount, top_k])]
        teacher_retriever_amount = 1 if current_epoch_number == 0 else len(teacher_retriever_pool)
        updated_query_data_batch = []
        for query_data in query_data_batch:
            updated_query_data_batch.append(
                (query_data[0], query_data[1][:teacher_retriever_amount])
            )
        return updated_query_data_batch
    
    def set_current_epoch_number(value: int):
        nonlocal current_epoch_number
        current_epoch_number = value
    
    return povaryoshka_retriever_collate_fn, set_current_epoch_number


In [21]:
def compute_recall_at_k(
    retriever_model: PovaryoshkaRetriever,
    query_list: list[str],
    true_index_list: list[int],
    k=5
) -> float:
    retriever_model.eval()
    correct_amount = 0
    with torch.no_grad():
        retrieved_indices_list = retriever_model.search(query_list, top_k=k)
        for i in range(len(query_list)):
            if true_index_list[i] in retrieved_indices_list[i]:
                correct_amount += 1
    return correct_amount / len(query_list)


def copy_data_to_device(data, device):
  if torch.is_tensor(data):
    return data.to(device)
  elif isinstance(data, (list, tuple)):
    return [copy_data_to_device(elem, device) for elem in data]
  return data


def loss_plot(losses: list[float]):
  plt.plot(np.arange(len(losses)), losses)
  plt.title('Losses')
  plt.xlabel("epoch")
  plt.show()


def retriever_train_loop(
    retriever_model: PovaryoshkaRetriever,
    set_current_epoch_number: Callable[[int], None],
    train_dataloader,
    eval_dataloader,
    lr=1e-4,
    epoch_amount=10,
    device: Literal['cuda', 'cpu', 'mps']='cuda',
    l2_reg_alpha=0,
    optimizer_ctor=None,
    lr_scheduler_ctor=None,
    early_stopping_patience=10,
    k=5
):
    if optimizer_ctor is None:
        optimizer = torch.optim.Adam(retriever_model.parameters(), lr=lr, weight_decay=l2_reg_alpha)
    else:
        optimizer = optimizer_ctor(retriever_model.parameters(), lr=lr)

    if lr_scheduler_ctor is not None:
        lr_scheduler = lr_scheduler_ctor(optimizer)

    best_eval_loss = float('inf')
    retriever_model = retriever_model.to(device)
    best_retriever_model = copy.deepcopy(retriever_model)
    retriever_model.train()
    loss_list = []

    for epoch_index in range(epoch_amount):
        try:
            set_current_epoch_number(epoch_index)
            epoch_start = datetime.datetime.now()
            print(f'Эпоха {epoch_index}')
            retriever_model.train()
            mean_train_loss = 0
            train_batches_n = 0
            for train_batch in train_dataloader:
                train_batch = copy_data_to_device(train_batch, device)
                train_loss = retriever_model(train_batch)
                retriever_model.zero_grad()
                train_loss.backward()
                optimizer.step()
                mean_train_loss += train_loss.item()
                train_batches_n += 1

            mean_train_loss /= train_batches_n
            loss_list.append(mean_train_loss)
            print('Эпоха: {} итераций, {:0.2f} сек'.format(train_batches_n,
                                                           (datetime.datetime.now() - epoch_start).total_seconds()))

            if eval_dataloader is None:
                if lr_scheduler is not None:
                    lr_scheduler.step(mean_train_loss)
                continue

            retriever_model.eval()
            mean_eval_loss = 0
            eval_batches_n = 0
            with torch.no_grad():
                for eval_batch in eval_dataloader:
                    eval_batch = copy_data_to_device(eval_batch, device)
                    eval_loss = retriever_model(eval_batch)
                    mean_eval_loss += float(eval_loss)
                    eval_batches_n += 1

            mean_eval_loss /= eval_batches_n
            print(f'Среднее значение функции потерь на валидации {mean_eval_loss}')

                            
            retriever_model.build_index()
            for eval_batch in eval_dataloader:
                eval_batch = copy_data_to_device(eval_batch, device)
                _, positive_document_index_tensor, _ = retriever_model.teacher_retriever_pool(
                    eval_batch,
                    device
                )
                recall_at_k = compute_recall_at_k(
                    retriever_model,
                    [chunk['questions'][0] for chunk in eval_ranked_chunk_list],
                    positive_document_index_tensor.cpu().numpy().tolist(),
                    k=k
                )
                print(f'Recall@{k}: {recall_at_k}')

            if mean_eval_loss < best_eval_loss:
                best_epoch_index = epoch_index
                best_eval_loss = mean_eval_loss
                best_retriever_model = copy.deepcopy(retriever_model)
                print('Новая лучшая модель!')
            elif epoch_index - best_epoch_index > early_stopping_patience:
                print('Модель не улучшилась за последние {} эпох, прекращаем обучение'.format(
                    early_stopping_patience))
                break
            print()
        except KeyboardInterrupt:
            print('Досрочно остановлено пользователем')
            break
        except Exception as ex:
            print(f'Ошибка при обучении: {ex}\n{traceback.format_exc()}')
            break
    loss_plot(loss_list)
    if eval_dataloader is None:
        return retriever_model, None
    return best_retriever_model, best_eval_loss
    

In [22]:
train_dataset = RetrieverDataset(train_ranked_chunk_list)
povaryoshka_retriever_collate_fn, set_current_epoch_number = create_povaryoshka_retriever_fn(
  teacher_retriever_pool
)
train_dataloader = DataLoader(
  train_dataset,
  batch_size=batch_size,
  shuffle=True,
  collate_fn=povaryoshka_retriever_collate_fn,
  num_workers=0
)
eval_dataset = RetrieverDataset(eval_ranked_chunk_list)
eval_dataloader = DataLoader(
  eval_dataset,
  batch_size=batch_size,
  shuffle=True,
  collate_fn=povaryoshka_retriever_collate_fn,
  num_workers=0
)


common_document_list = [chunk['text'] for chunk in pruned_common_chunk_list]
llm_model = PovaryoshkaRetriever(
  common_document_list,
  teacher_retriever_pool,
  temperature=0.1,
  dtype='float32',
  nlist=1
)

scheduler = lambda optim: \
    torch.optim.lr_scheduler.ReduceLROnPlateau(optim, patience=3, factor=0.43)
trained_retriever_model, _ = retriever_train_loop(
  llm_model,
  set_current_epoch_number,
  train_dataloader,
  eval_dataloader,
  lr_scheduler_ctor=scheduler,
  epoch_amount=10,
  device=device
)

torch.save(trained_retriever_model.state_dict(), 'povaryoshka_retriever_model_weights.pth')
print("Модель сохранена!")

Compiling the model with `torch.compile` and using a `torch.cpu` device is not supported. Falling back to non-compiled mode.


Эпоха 0
Эпоха: 2 итераций, 3.38 сек
Среднее значение функции потерь на валидации 6.095210075378418
float32
📊 Индексация 20 документов...


: 

In [ ]:
# teacher_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
teacher_model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    teacher_model_name,
    device_map="auto",
    dtype=torch.float16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
def generate_answer(prompt):
  inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)

  # Генерация текста
  with torch.no_grad():  # отключаем градиенты для inference
    outputs = llm_model.generate(
      **inputs,
      max_new_tokens=100,  
      do_sample=True,      
      temperature=0.7,     
      top_p=0.9            
  )

  # Превращаем токены обратно в текст
  generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

  return generated_text

In [ ]:
def generate_until_eos(prompt, max_total_tokens=1024, temperature=0.7):
    """
    Генерация текста токен за токеном до конца (EOS) или пока не достигнем max_total_tokens.
    """
    llm_model.eval()
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(llm_model.device)
    generated_ids = input_ids

    with torch.no_grad():
        for _ in range(max_total_tokens):
            # Берём последние N токенов как context (N = context_length модели)
            output = llm_model(generated_ids)
            logits = output.logits

            # Берём последний токен
            next_token_logits = logits[:, -1, :] / temperature
            probs = torch.softmax(next_token_logits, dim=-1)

            # Сэмплируем следующий токен
            next_token = torch.multinomial(probs, num_samples=1)
            generated_ids = torch.cat([generated_ids, next_token], dim=1)

            # Если токен EOS → прекращаем генерацию
            if next_token.item() == tokenizer.eos_token_id:
                print(next_token.item())
                break

    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
question_prompt = "Explain how to cook pasta"

print(generate_until_eos(question_prompt))

Explain how to cook pasta to make it al dente, and provide tips for achieving the perfect texture and flavor.


In [ ]:
def rag_answer(query, retriever):

    retrieved_docs = retriever.search(query)

    context = "\n".join(retrieved_docs)

    prompt = f"""
Контекст:
{context}

Вопрос:
{query}
"""

    return generate_until_eos(prompt)

text_chunk_list = [chunk['text'] for chunk in train_chunk_list]
trained_model.build_index(text_chunk_list[:1000])

In [76]:
print(rag_answer("Как приготовить борщ?", trained_model))


Контекст:
Инструкции для рецепта "":
1) Сварить из костей наваристый бульон, отдельно до готовности – перловку.

Инструкции для рецепта "":
1) Налить в кастрюлю воду для супа, довести до кипения, подсолить, добавить морковь и картофель, нарезанные небольшими кубиками, варить до мягкости.

Инструкции для рецепта "":
3) Перед подачей татарский суп с мясом сдобрить большим количеством рубленой петрушки и заправить измельченным чесноком.


Вопрос:
Как приготовить борщ?

Ответ:
1) Лист печени куба (кожа) вручную разрезать вдоль гребней.

2) В очень холодном озёре поместить лист печени, листья и стручки ореха, слить в какой-то чемодан смотровую стяжку, заморозить и высушить в течение 1-2 часов.

3) В кастрюлю воды вкушить целый лист печени, слишить из чемодана и перемешать с орехами и крушинами.

4) Дать орехи и крушины в кипяницу, вкушить и оставаясь в кипящем орехе и крушине, добавить небольшое количество масла и тушать до получения мягкой творожки.

5) Перемешать в кастрюлю воду для супа

In [21]:
import torch

# =========================
# 🧠 RAG система
# =========================
class RAGSystem:
    def __init__(self, retriever, generator_fn):
        self.retriever = retriever
        self.generate = generator_fn

    def answer(self, query, top_k=3):
        # 1. Достаем релевантные документы
        docs = self.retriever.search(query, top_k=top_k)

        # 2. Собираем контекст
        context = "\n".join(docs)

        # 3. Формируем prompt
        prompt = f"""
You are a helpful cooking assistant.

Use the context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

        # 4. Генерация
        return self.generate(prompt)


# =========================
# 🖥️ CLI интерфейс
# =========================
def chat_loop(rag_system):
    print("🔥 RAG готов! Введи вопрос (exit чтобы выйти)\n")

    while True:
        query = input("👤 Ты: ")

        if query.lower() in ["exit", "quit"]:
            print("Пока 👋")
            break

        answer = rag_system.answer(query)

        print("\n🤖 Ответ:")
        print(answer)
        print("\n" + "="*50 + "\n")

In [22]:
# 1. создаешь retriever
# retriever = PovaryoshkaRetriever()

# # 2. индексируешь документы
# docs = [
#     "Boil pasta in salted water for 10 minutes.",
#     "Add olive oil and garlic for better taste.",
#     "Tomato sauce is commonly used for pasta."
# ]

# retriever.build_index(docs)

# 3. создаешь RAG
rag = RAGSystem(trained_model, generate_until_eos)

# 4. запускаешь чат
chat_loop(rag)

🔥 RAG готов! Введи вопрос (exit чтобы выйти)

👤 Ты: Хочу приготовить завтра утром блинчики. Как их приготовить ?

🤖 Ответ:

You are a helpful cooking assistant.

Use the context to answer the question.

Context:
Инструкции для рецепта "":
2) Печь в духовке до затвердевания, снять, остудить. Для приготовления начинки протереть творог через сито, смешать со сливками с сахаром, взбитыми в крепкую пену. В массу добавить цедру лимона или апельсина, ванилин. Наполнить творогом трубочки из блинчиков, полить растопленным на водяной бане шоколадом. Подать сразу.

Инструкции для рецепта "":
1) Выпечь полтора десятка тонких блинчиков. Формочки для трубочек смазать растопленным сливочным маслом, обернуть готовыми блинчиками внахлест. Чтобы блинчики лучше держались в месте соединения, можно смазать их чуть взбитым яйцом или остатками теста. Излишки обрезать по форме конуса.

Инструкции для рецепта "":
2) Выложить тесто в формы для выпечки, на две трети объема. Печь в духовке в течение получаса, гот

KeyboardInterrupt: Interrupted by user

In [ ]:
import torch
from torch.utils.data import Dataset

class RAGDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=1024):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def build_prompt(self, query, docs, answer):
        context = "\n".join(docs)
        prompt = f"""### Вопрос:
{query}

### Контекст:
{context}

### Ответ:
{answer}"""
        return prompt

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        prompt = self.build_prompt(
            item["query"],
            item["docs"],
            item["answer"]
        )

        tokens = self.tokenizer(
            prompt,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = tokens["input_ids"].squeeze(0)
        attention_mask = tokens["attention_mask"].squeeze(0)

        # labels = input_ids (но с маской!)
        labels = input_ids.clone()

        # 🔥 маскируем всё ДО ответа
        answer_start = prompt.find("### Ответ:")
        answer_tokens = self.tokenizer(
            prompt[:answer_start],
            return_tensors="pt"
        )["input_ids"].shape[1]

        labels[:answer_tokens] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    labels = [item["labels"] for item in batch]

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
from torch.utils.data import DataLoader

dataset = RAGDataset(data, tokenizer)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

teacher_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    teacher_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 🔥 LoRA
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none"
)

llm_model = get_peft_model(llm_model, config)
llm_model.train()

In [ ]:
import torch
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(llm_model.parameters(), lr=2e-5)

device = next(llm_model.parameters()).device

epoch_amount = 3

for epoch in range(epoch_amount):
    total_loss = 0

    for query_data_batch in tqdm(dataloader):
        input_ids = query_data_batch["input_ids"].to(device)
        attention_mask = query_data_batch["attention_mask"].to(device)
        labels = query_data_batch["labels"].to(device)

        model_output = llm_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = model_output.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: loss = {total_loss / len(dataloader)}")

In [ ]:
llm_model.eval()

query_list = "Как сварить борщ?"
docs = ["Свеклу варят...", "Добавляют картошку..."]

question_prompt = f"""### Вопрос:
{query_list}

### Контекст:
{chr(10).join(docs)}

### Ответ:
"""

tokenizer_output = tokenizer(question_prompt, return_tensors="pt").to(device)

output = llm_model.generate(
    **tokenizer_output,
    max_new_tokens=100,
    do_sample=True
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Teacher model
teacher_model_name = "meta-llama/Llama-2-7b-chat-hf"  # можно GPT-4 через API
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)
teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

question_generator = pipeline(
    "text-generation",
    model=teacher_model,
    tokenizer=teacher_tokenizer,
    device=0
)

def generate_target(query, docs):
    context = "\n".join(docs)
    prompt = f"""
Ответь на вопрос, используя только контекст ниже.
Если ответа нет в контексте, скажи "не знаю".

Вопрос: {query}

Контекст:
{context}

Ответ:
"""
    output = question_generator(prompt, max_new_tokens=150, do_sample=False)
    answer = output[0]["generated_text"].split("Ответ:")[-1].strip()
    return answer

In [ ]:
query_list = "Как сварить борщ?"
docs = ["Свеклу варят...", "Добавляют картошку...", "Борщ готовится..."]

target_answer = generate_target(query_list, docs)
print(target_answer)

In [13]:
import torch

a = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])
print(a.transpose(0, 1))

tensor([[1, 4],
        [2, 5],
        [3, 6]])
